# VinDr-SpineXR 임플란트(고밀도 이물질) 탐지 — YOLO Colab 학습·평가 및 ONNX 검증

`scripts/prepare_spinexr.py convert`로 생성한 `data/vindr-yolo/` 데이터셋을 사용해 **Google Colab GPU**에서 YOLO 모델을 학습하고,
가중치(`best.pt`, `best.onnx`)와 평가 지표(**Precision, Recall, F1, mAP@0.5, 영상당 오검출 수(FP/image)**)를 산출하는 노트북입니다.

### 평가 프로토콜 (비교 문서 기준과 100% 통일)
- **IoU 상수 분리·통일**: 추론/ONNX 후처리 NMS는 `NMS_IOU = 0.45`, GT 1:1 매칭 엄격 평가는 `MATCH_IOU = 0.50`
- **임계값 선정·고정 (test 누수 방지)**: `val` 세트(100장)에서 threshold sweep을 먼저 수행해 최고 F1 기준 임계값(`SELECTED_CONF`, 실측 `0.40`)을 선정·고정한 뒤 `test` 세트(122장) 공식 최종 지표를 계산
- **`test` 전체 구간 sweep**: 민감도 확인용 **탐색 참고 결과(post-hoc exploratory)**로 별도 구분
- **ONNX 변환 및 일치 검증**: `best.onnx` (`opset=18`, `imgsz=640`, NMS 미포함) 변환 후 순수 `onnxruntime`(`iou_thres=NMS_IOU=0.45`)과 PyTorch 동일 입력 일치 검증 수행

### 사전 준비 (로컬에서 실행)
1. `scripts/prepare_spinexr.py plan` 및 `convert` 실행 → `data/vindr-yolo/` 생성
2. `data/vindr-yolo` 폴더를 zip으로 압축:
   ```bash
   cd data && zip -r vindr-yolo.zip vindr-yolo
   ```
3. 압축한 `vindr-yolo.zip`을 본인 **Google Drive (비공개)** `MyDrive/vindr-spinexr/vindr-yolo.zip`에 업로드
4. Colab 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU (또는 L4/A100)** 선택

> ⚠️ **데이터 라이선스 주의**: VinDr-SpineXR은 PhysioNet Restricted Health Data License 1.5.0이 적용됩니다.
> Google Drive에 올린 데이터와 zip 파일은 **본인만 접근(비공개)**으로 유지하고, GitHub 저장소에 커밋하거나 외부 공유하지 마세요.
> 본 모델 결과는 교육용이며 실제 진단 용도가 아닙니다.

## 1. GPU 환경 확인 및 패키지 설치

In [ ]:
import subprocess
import sys

# GPU 확인
res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    print("⚠️ GPU가 감지되지 않았습니다. [런타임 → 런타임 유형 변경 → GPU]를 선택하세요.")

# Ultralytics, ONNX Runtime 및 보조 패키지 설치
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pyyaml", "pandas", "matplotlib", "onnx", "onnxruntime"]
)

import onnxruntime as ort
import torch
import ultralytics

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics: {ultralytics.__version__} | ONNX Runtime: {ort.__version__}")

## 2. Google Drive 마운트 및 경로·하이퍼파라미터 설정

- 로컬에서 `prepare_spinexr.py convert` 기본 `--max-size 1024`로 변환했다면 `IMGSZ = 640` (기본 베이스라인) 또는 `IMGSZ = 1024` (차기 고해상도 비교 후보)를 설정할 수 있습니다.
- **평가·후처리 상수 통일**:
  - `NMS_IOU = 0.45`: 추론(`model.predict` / `model.val`) 및 ONNX 후처리 NMS 기준
  - `MATCH_IOU = 0.50`: 정답 박스(GT) 1:1 매칭 엄격 평가 기준 (`IoU >= 0.50`)
  - `DEFAULT_CONF = 0.40`: `val` 세트(100장) 최고 F1 실측 기준 기본 임계값
- 학습 완료 후 가중치(`best.pt`, `best.onnx`)와 평가 지표는 `DRIVE_SAVE_DIR`에 자동 백업됩니다.

In [ ]:
from pathlib import Path
import shutil
import zipfile

# Google Drive 마운트 (Colab 환경에서만 실행)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ======================== 사용자 설정 ========================
# 1) 데이터 경로 (zip 파일 또는 이미 압축 해제된 폴더 중 존재하는 경로 사용)
DRIVE_ZIP_PATH = Path("/content/drive/MyDrive/vindr-spinexr/vindr-yolo.zip")
LOCAL_DATA_DIR = Path("/content/vindr-yolo")  # 로컬 SSD에 풀어야 학습 속도가 빠름

# 2) 결과 백업 경로 (Google Drive)
RUN_NAME = "spinexr_implant_yolo11n_640"
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/vindr-spinexr/runs") / RUN_NAME

# 3) 학습 하이퍼파라미터
MODEL_NAME = "yolo11n.pt"   # 후보: "yolo11n.pt", "yolo11s.pt", "yolov8n.pt"
EPOCHS = 50
IMGSZ = 640                 # 640 (기본 베이스라인) 또는 1024 (고해상도 비교 후보)
BATCH = 16                  # OOM 발생 시 8로 줄이기
PATIENCE = 15
SEED = 42

# 4) 평가·후처리 및 ONNX 변환 설정 (NMS IoU와 GT 매칭 IoU 분리·통일)
NMS_IOU = 0.45              # 추론 및 ONNX 후처리 NMS 기준
MATCH_IOU = 0.50            # GT 1:1 매칭 엄격 평가 기준 (IoU >= 0.50)
DEFAULT_CONF = 0.40         # val 최고 F1 선정·고정 기본 임계값 (실측 0.40)
CONF_SWEEP = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
ONNX_OPSET = 18
# =============================================================

# 압축 해제 (이미 풀려 있으면 건너뜀)
if not (LOCAL_DATA_DIR / "images").exists():
    if not DRIVE_ZIP_PATH.exists():
        raise FileNotFoundError(
            f"데이터 zip 파일을 찾을 수 없습니다: {DRIVE_ZIP_PATH}\n"
            "로컬에서 'cd data && zip -r vindr-yolo.zip vindr-yolo' 실행 후 Drive에 업로드하세요."
        )
    extract_root = LOCAL_DATA_DIR.parent
    extract_root.mkdir(parents=True, exist_ok=True)
    print(f"압축 해제 중: {DRIVE_ZIP_PATH} -> {extract_root}")
    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as zf:
        zf.extractall(extract_root)

    # zip 내부 최상위 폴더 이름이 vindr-yolo가 아닌 경우 자동 탐색
    if not (LOCAL_DATA_DIR / "images").exists():
        candidates = list(extract_root.rglob("dataset.yaml"))
        if candidates:
            LOCAL_DATA_DIR = candidates[0].parent
            print(f"dataset.yaml 위치 자동 감지: {LOCAL_DATA_DIR}")
        else:
            raise FileNotFoundError("압축 해제 결과에서 dataset.yaml / images 폴더를 찾지 못했습니다.")

print(f"사용할 데이터 경로: {LOCAL_DATA_DIR}")
print(f"설정 확인: NMS_IOU={NMS_IOU}, MATCH_IOU={MATCH_IOU}, DEFAULT_CONF={DEFAULT_CONF}, ONNX_OPSET={ONNX_OPSET}")

## 3. `dataset.yaml` Colab 경로 보정 및 데이터셋 분포 점검

`scripts/prepare_spinexr.py convert`가 생성한 `dataset.yaml`에는 로컬 절대 경로(`path: /Users/.../data/vindr-yolo`)가 기록되어 있습니다.
아래 셀은 이를 현재 Colab 경로(`/content/vindr-yolo`)로 자동 보정한 `dataset_colab.yaml`을 만들고, `train / val / test` 분할별 임플란트(양성)·정상(음성) 영상 수와 박스 수를 검증합니다.

In [ ]:
import pandas as pd
import yaml

orig_yaml = LOCAL_DATA_DIR / "dataset.yaml"
if orig_yaml.exists():
    cfg = yaml.safe_load(orig_yaml.read_text(encoding="utf-8"))
else:
    cfg = {
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "implant"},
    }

# Colab 경로로 path 보정
cfg["path"] = LOCAL_DATA_DIR.resolve().as_posix()
COLAB_YAML = LOCAL_DATA_DIR / "dataset_colab.yaml"
COLAB_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(f"생성 완료: {COLAB_YAML}")
print(COLAB_YAML.read_text(encoding="utf-8"))

# 분할별 이미지·라벨 무결성 및 양성/정상 분포 점검
IMG_EXTS = {".png", ".jpg", ".jpeg"}
summary_rows = []
for split in ("train", "val", "test"):
    img_dir = LOCAL_DATA_DIR / "images" / split
    lbl_dir = LOCAL_DATA_DIR / "labels" / split
    images = sorted(p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXTS) if img_dir.exists() else []

    n_implant = 0
    n_normal = 0
    n_missing_lbl = 0
    n_boxes = 0
    for img_path in images:
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            n_missing_lbl += 1
            continue
        lines = [ln.strip() for ln in lbl_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if lines:
            n_implant += 1
            n_boxes += len(lines)
        else:
            n_normal += 1

    summary_rows.append({
        "split": split,
        "total_images": len(images),
        "implant_images": n_implant,
        "normal_images": n_normal,
        "missing_labels": n_missing_lbl,
        "total_boxes": n_boxes,
    })

df_data = pd.DataFrame(summary_rows)
display(df_data)

if df_data.loc[df_data["split"] == "train", "total_images"].iloc[0] == 0:
    raise RuntimeError("train 이미지가 0장입니다. 데이터 압축 경로를 확인하세요.")
if df_data["missing_labels"].sum() > 0:
    print("⚠️ 일부 이미지의 라벨 파일(.txt)이 누락되었습니다. convert 결과를 확인하세요.")

## 4. YOLO 모델 학습

- `fliplr=0.5`: 척추 X-ray 좌우 반전 증강 허용
- `flipud=0.0`: 상하 반전은 해부학적 방향이 뒤집히므로 비활성화
- `hsv_h=0.0, hsv_s=0.0`: 흑백 X-ray이므로 색상/채도 증강은 끄고 밝기(`hsv_v=0.2`)만 약하게 적용

In [ ]:
from ultralytics import YOLO

TRAIN_PROJECT = Path("/content/runs/detect")
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(COLAB_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(TRAIN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    # 의료 흑백 X-ray에 맞춘 증강 설정
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    degrees=5.0,
    fliplr=0.5,
    flipud=0.0,
    plots=True,
    save=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print(f"학습 결과 폴더: {RUN_DIR}")
print(f"최적 가중치 파일: {BEST_WEIGHTS} (exists={BEST_WEIGHTS.exists()})")

## 5. 공식 평가 지표 산출 (`val` & `test` — Precision, Recall, mAP@0.5)

`best.pt`를 불러와 `val`과 `test` 분할에 대해 Ultralytics 표준 평가(`model.val`)를 실행합니다.
Ultralytics `val()`의 `iou` 인자는 **추론 NMS IoU 임계값**이므로 `iou=NMS_IOU` (`0.45`)로 통일하여 전달합니다 (mAP@0.5의 GT 매칭 기준은 내부 0.50 고정).

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))

official_metrics = {}
for split in ("val", "test"):
    split_dir = LOCAL_DATA_DIR / "images" / split
    if not split_dir.exists() or not any(split_dir.iterdir()):
        print(f"[{split}] 이미지가 없어 건너뜁니다.")
        continue
    m = best_model.val(
        data=str(COLAB_YAML),
        split=split,
        imgsz=IMGSZ,
        batch=BATCH,
        iou=NMS_IOU,  # Ultralytics val()의 iou 인자는 NMS IoU 기준 (0.45)
        device=0 if torch.cuda.is_available() else "cpu",
        project=str(TRAIN_PROJECT),
        name=f"{RUN_NAME}_eval_{split}",
        exist_ok=True,
        plots=True,
    )
    official_metrics[split] = {
        "precision": round(float(m.box.mp), 4),
        "recall": round(float(m.box.mr), 4),
        "mAP50": round(float(m.box.map50), 4),
        "mAP50_95": round(float(m.box.map), 4),
    }

print(f"\n=== Ultralytics 공식 평가 결과 (NMS IoU={NMS_IOU}) ===")
display(pd.DataFrame(official_metrics).T)

## 6. `val` 기준 임계값 선정·고정(`SELECTED_CONF`) 및 `test` 공식 최종 평가 / 탐색 참고 스윕

Ultralytics `val()` 기본 출력에는 **영상당 오검출 수(FP/image)**와 정상 영상 오탐률(`normal FPR`)이 직접 표시되지 않습니다.
아래 코드는 **추론 NMS `NMS_IOU = 0.45`**, **GT 1:1 매칭 `MATCH_IOU = 0.50`** 기준으로 다음 순서에 따라 엄격히 평가합니다:

1. **1단계 (`val` 100장 threshold sweep → `SELECTED_CONF` 선정·고정)**: `val` 세트에서 `CONF_SWEEP` 전체를 평가해 최고 F1 기준 임계값(`SELECTED_CONF`, 실측 `0.40`)을 선정·고정합니다.
2. **2단계 (`test` 122장 공식 최종 평가)**: 선정·고정된 `SELECTED_CONF`만 적용해 미사용 평가 세트(`test`)의 공식 최종 지표를 산출합니다 (임계값 선택 편향 / test 누수 방지).
3. **3단계 (`test` 전체 구간 스윕 — Post-hoc Exploratory)**: `test` 전체 임계값 구간 스윕은 민감도 확인을 위한 **탐색 참고 결과(post-hoc exploratory)**로만 별도 표기하며, 모델/임계값 선택 기준으로 사용하지 않습니다.

산출 지표:
- **Precision / Recall / F1** (GT 1:1 greedy 매칭, `IoU >= MATCH_IOU`)
- **FP/image (전체)**: 전체 평가 영상 1장당 평균 오검출 박스 수 (`FP / 전체 영상 수`)
- **FP/normal (정상)**: 정상(`No finding`, GT=0) 영상 1장당 평균 오검출 박스 수
- **정상 오탐률 (`normal_FPR`)**: 정상 영상 중 1개 이상 오검출이 발생한 영상 비율
- **FP/implant (임플란트)**: 임플란트(양성) 영상 1장당 평균 오검출 박스 수

In [ ]:
import cv2
import numpy as np


def yolo_txt_to_xyxy(label_path: Path, img_w: int, img_h: int) -> np.ndarray:
    """YOLO 정규화 라벨(.txt)을 픽셀 단위 [x1, y1, x2, y2] 배열로 변환한다."""
    if not label_path.exists():
        return np.zeros((0, 4), dtype=np.float32)
    boxes = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        _, cx, cy, w, h = map(float, parts[:5])
        x1 = (cx - w / 2.0) * img_w
        y1 = (cy - h / 2.0) * img_h
        x2 = (cx + w / 2.0) * img_w
        y2 = (cy + h / 2.0) * img_h
        boxes.append([x1, y1, x2, y2])
    return np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)


def compute_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """두 박스 집합 (N,4), (M,4) 간의 IoU 행렬 (N,M)을 계산한다."""
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    x1 = np.maximum(boxes_a[:, None, 0], boxes_b[None, :, 0])
    y1 = np.maximum(boxes_a[:, None, 1], boxes_b[None, :, 1])
    x2 = np.minimum(boxes_a[:, None, 2], boxes_b[None, :, 2])
    y2 = np.minimum(boxes_a[:, None, 3], boxes_b[None, :, 3])
    inter = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)
    area_a = np.maximum(0.0, boxes_a[:, 2] - boxes_a[:, 0]) * np.maximum(0.0, boxes_a[:, 3] - boxes_a[:, 1])
    area_b = np.maximum(0.0, boxes_b[:, 2] - boxes_b[:, 0]) * np.maximum(0.0, boxes_b[:, 3] - boxes_b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)


def collect_predictions(
    model_obj: YOLO,
    data_dir: Path,
    split: str,
    imgsz: int,
    nms_iou: float = NMS_IOU,
) -> list[dict]:
    """split 전체 영상에 대해 낮은 conf(0.001)와 NMS_IOU(0.45)로 1회 추론해 GT와 예측 박스를 모은다."""
    img_dir = data_dir / "images" / split
    lbl_dir = data_dir / "labels" / split
    img_paths = sorted(p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXTS)
    records = []
    for img_path in img_paths:
        res = model_obj.predict(
            source=str(img_path),
            imgsz=imgsz,
            conf=0.001,
            iou=nms_iou,
            device=0 if torch.cuda.is_available() else "cpu",
            verbose=False,
        )[0]
        img_h, img_w = res.orig_shape
        gt_boxes = yolo_txt_to_xyxy(lbl_dir / f"{img_path.stem}.txt", img_w, img_h)
        if res.boxes is not None and len(res.boxes) > 0:
            pred_boxes = res.boxes.xyxy.cpu().numpy().astype(np.float32)
            pred_confs = res.boxes.conf.cpu().numpy().astype(np.float32)
        else:
            pred_boxes = np.zeros((0, 4), dtype=np.float32)
            pred_confs = np.zeros((0,), dtype=np.float32)
        records.append({
            "image_id": img_path.stem,
            "image_path": img_path,
            "gt_boxes": gt_boxes,
            "pred_boxes": pred_boxes,
            "pred_confs": pred_confs,
        })
    return records


def evaluate_at_conf(
    records: list[dict],
    conf_thresh: float,
    iou_thresh: float = MATCH_IOU,
) -> tuple[dict, list[dict]]:
    """지정 conf 및 GT 매칭 기준(MATCH_IOU=0.50)으로 TP, FP, FN, Precision, Recall, F1, 영상당 오검출 수를 계산한다."""
    total_tp = total_fp = total_fn = 0
    n_normal = n_normal_fp_boxes = n_normal_fp_imgs = 0
    n_implant = n_implant_fp_boxes = 0
    per_image = []

    for rec in records:
        gt = rec["gt_boxes"]
        keep = rec["pred_confs"] >= conf_thresh
        preds = rec["pred_boxes"][keep]
        confs = rec["pred_confs"][keep]

        # confidence 내림차순 1:1 greedy 매칭
        order = np.argsort(-confs)
        preds = preds[order]
        confs = confs[order]

        matched_gt = set()
        tp = fp = 0
        iou_mat = compute_iou_matrix(preds, gt)
        for pi in range(len(preds)):
            if len(gt) == 0:
                fp += 1
                continue
            best_gi = int(np.argmax(iou_mat[pi]))
            if iou_mat[pi, best_gi] >= iou_thresh and best_gi not in matched_gt:
                matched_gt.add(best_gi)
                tp += 1
            else:
                fp += 1
        fn = len(gt) - len(matched_gt)

        total_tp += tp
        total_fp += fp
        total_fn += fn

        is_normal = len(gt) == 0
        if is_normal:
            n_normal += 1
            n_normal_fp_boxes += fp
            if fp > 0:
                n_normal_fp_imgs += 1
        else:
            n_implant += 1
            n_implant_fp_boxes += fp

        per_image.append({
            "image_id": rec["image_id"],
            "image_path": rec["image_path"],
            "is_normal": is_normal,
            "tp": tp, "fp": fp, "fn": fn,
            "gt_boxes": gt, "pred_boxes": preds, "pred_confs": confs,
        })

    n_total = len(records)
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    metrics = {
        "conf": conf_thresh,
        "TP": total_tp,
        "FP": total_fp,
        "FN": total_fn,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "FP_per_image_all": round(total_fp / n_total, 4) if n_total else 0.0,
        "FP_per_normal_image": round(n_normal_fp_boxes / n_normal, 4) if n_normal else 0.0,
        "normal_FPR": round(n_normal_fp_imgs / n_normal, 4) if n_normal else 0.0,
        "FP_per_implant_image": round(n_implant_fp_boxes / n_implant, 4) if n_implant else 0.0,
    }
    return metrics, per_image


# 1) val 세트에서 threshold sweep을 먼저 수행하여 최고 F1 기준 임계값(SELECTED_CONF)을 선정·고정
val_img_dir = LOCAL_DATA_DIR / "images" / "val"
has_val = val_img_dir.exists() and any(val_img_dir.iterdir())

if has_val:
    print(f"[1단계 - val] 전체 추론 및 임계값 선정 스윕 시작 (NMS_IOU={NMS_IOU}, MATCH_IOU={MATCH_IOU})...")
    val_records = collect_predictions(best_model, LOCAL_DATA_DIR, "val", IMGSZ, nms_iou=NMS_IOU)
    val_sweep_rows = []
    for c in CONF_SWEEP:
        row, _ = evaluate_at_conf(val_records, conf_thresh=c, iou_thresh=MATCH_IOU)
        val_sweep_rows.append(row)
    df_val_sweep = pd.DataFrame(val_sweep_rows)
    val_best_row = df_val_sweep.loc[df_val_sweep["F1"].idxmax()].to_dict()
    SELECTED_CONF = float(val_best_row["conf"])
    df_val_sweep["role"] = np.where(
        (df_val_sweep["conf"] - SELECTED_CONF).abs() < 1e-9,
        "최고 F1 선정 (고정)",
        "후보",
    )
    print(f"\n=== [val] 임계값 선정 스윕 결과 (N={len(val_records)}, NMS IoU={NMS_IOU}, Match IoU={MATCH_IOU}) ===")
    display(df_val_sweep)
    print(
        f"\n✅ val 최고 F1 기준 임계값 선정·고정: SELECTED_CONF={SELECTED_CONF:.2f} "
        f"(DEFAULT_CONF={DEFAULT_CONF:.2f}, val F1={val_best_row['F1']}, "
        f"P={val_best_row['Precision']}, R={val_best_row['Recall']}, "
        f"FP/norm={val_best_row['FP_per_normal_image']}, normal_FPR={val_best_row['normal_FPR']:.2%})"
    )
else:
    print(f"⚠️ val 분할이 없어 DEFAULT_CONF={DEFAULT_CONF:.2f}를 SELECTED_CONF로 사용합니다.")
    val_records = []
    df_val_sweep = pd.DataFrame()
    val_best_row = {}
    SELECTED_CONF = DEFAULT_CONF

# 2) 고정된 SELECTED_CONF로 test 세트 공식 최종 지표 계산
test_img_dir = LOCAL_DATA_DIR / "images" / "test"
EVAL_SPLIT = "test" if test_img_dir.exists() and any(test_img_dir.iterdir()) else "val"
print(f"\n[2단계 - {EVAL_SPLIT}] 고정 임계값(SELECTED_CONF={SELECTED_CONF:.2f}) 공식 최종 평가 시작...")
test_records = collect_predictions(best_model, LOCAL_DATA_DIR, EVAL_SPLIT, IMGSZ, nms_iou=NMS_IOU)
test_official_row, per_image_at_selected = evaluate_at_conf(
    test_records, conf_thresh=SELECTED_CONF, iou_thresh=MATCH_IOU
)

print(f"\n=== [{EVAL_SPLIT}] 공식 최종 지표 (val 선정·고정 conf={SELECTED_CONF:.2f}, NMS IoU={NMS_IOU}, Match IoU={MATCH_IOU}) ===")
print(f"- 공식 mAP@0.5: {official_metrics.get(EVAL_SPLIT, {}).get('mAP50')}")
print(
    f"- conf={SELECTED_CONF:.2f} (고정): TP={test_official_row['TP']}, FP={test_official_row['FP']}, FN={test_official_row['FN']} | "
    f"Precision={test_official_row['Precision']}, Recall={test_official_row['Recall']}, F1={test_official_row['F1']}"
)
print(
    f"- FP/image(전체)={test_official_row['FP_per_image_all']}, "
    f"FP/normal={test_official_row['FP_per_normal_image']} (normal FPR={test_official_row['normal_FPR']:.2%}), "
    f"FP/implant={test_official_row['FP_per_implant_image']}"
)

# 3) test 전체 구간 스윕 — 탐색 참고 결과 (Post-hoc Exploratory)
test_sweep_rows = []
for c in CONF_SWEEP:
    row, _ = evaluate_at_conf(test_records, conf_thresh=c, iou_thresh=MATCH_IOU)
    row["role"] = (
        "최종 고정 평가 (val 선정)"
        if abs(c - SELECTED_CONF) < 1e-9
        else "탐색 참고 (post-hoc exploratory)"
    )
    test_sweep_rows.append(row)

df_test_sweep = pd.DataFrame(test_sweep_rows)
test_posthoc_best_row = df_test_sweep.loc[df_test_sweep["F1"].idxmax()].to_dict()

print(
    f"\n=== [{EVAL_SPLIT}] Confidence 전체 구간 스윕 — 탐색 참고 결과 (Post-hoc Exploratory, "
    f"N={len(test_records)}, NMS IoU={NMS_IOU}, Match IoU={MATCH_IOU}) ==="
)
display(df_test_sweep)
print(
    f"(참고/post-hoc exploratory) {EVAL_SPLIT} 내 최대 F1 구간: conf={test_posthoc_best_row['conf']:.2f}, "
    f"F1={test_posthoc_best_row['F1']} (임계값 선택 기준으로 사용하지 않음)"
)

## 7. 정성 평가 및 오류 분석 시각화 (성공 / 미검출 FN / 정상 오검출 FP)

`val`에서 선정·고정한 `SELECTED_CONF` (실측 `0.40`) 및 `NMS_IOU = 0.45` 기준 예측 결과를 시각화합니다.
초록색 박스는 정답(GT), 빨간색 박스는 모델 예측(Pred + confidence)입니다.

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt


def plot_cases(cases: list[dict], title: str, save_path: Path | None = None):
    if not cases:
        print(f"{title}: 해당 사례가 없습니다.")
        return
    n = len(cases)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    for ax, item in zip(axes, cases):
        img = cv2.imread(str(item["image_path"]), cv2.IMREAD_GRAYSCALE)
        ax.imshow(img, cmap="gray")
        for x1, y1, x2, y2 in item["gt_boxes"]:
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none"))
        for (x1, y1, x2, y2), cf in zip(item["pred_boxes"], item["pred_confs"]):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none"))
            ax.text(x1, max(0, y1 - 4), f"{cf:.2f}", color="yellow", fontsize=9, bbox=dict(facecolor="red", alpha=0.6, pad=1))
        ax.set_title(f"{item['image_id'][:10]}...\nTP={item['tp']} FP={item['fp']} FN={item['fn']}", fontsize=10)
        ax.axis("off")
    fig.suptitle(title + " (Green=GT, Red=Pred)", fontsize=13)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


if per_image_at_selected:
    tp_cases = [x for x in per_image_at_selected if not x["is_normal"] and x["tp"] > 0 and x["fn"] == 0][:4]
    fn_cases = [x for x in per_image_at_selected if x["fn"] > 0][:4]
    fp_normal_cases = [x for x in per_image_at_selected if x["is_normal"] and x["fp"] > 0][:4]

    plot_cases(tp_cases, f"Implant TP Success (conf={SELECTED_CONF:.2f}, NMS={NMS_IOU})", RUN_DIR / "viz_tp_success.png")
    plot_cases(fn_cases, f"Implant Missed FN (conf={SELECTED_CONF:.2f}, NMS={NMS_IOU})", RUN_DIR / "viz_fn_missed.png")
    plot_cases(fp_normal_cases, f"Normal False Positive (conf={SELECTED_CONF:.2f}, NMS={NMS_IOU})", RUN_DIR / "viz_fp_normal.png")

## 8. ONNX 변환 (`best.onnx`, opset 18, `imgsz=640`) 및 PyTorch vs ONNX Runtime 일치 검증

로컬 CPU/Streamlit 환경에서 PyTorch 없이 경량 추론 배포가 가능하도록 `best.pt`를 `best.onnx` (`opset=18`, `imgsz=640`, 출력 `[1, 5, 8400]` NMS 미포함)로 변환하고,
순수 `onnxruntime` + NumPy 후처리(`conf_thres=SELECTED_CONF`, `iou_thres=NMS_IOU=0.45`)와 PyTorch 추론(`conf=SELECTED_CONF`, `iou=NMS_IOU=0.45`)의 동일 입력 일치 여부를 검증합니다.

In [ ]:
# 1) ONNX 변환 (opset 18, imgsz=640, 고정 배치 1, NMS 미포함)
exported_onnx_path = best_model.export(
    format="onnx",
    imgsz=IMGSZ,
    opset=ONNX_OPSET,
    dynamic=False,
    simplify=False,
)
BEST_ONNX = Path(exported_onnx_path)
onnx_size_mb = BEST_ONNX.stat().st_size / (1024 * 1024)

ort_session = ort.InferenceSession(str(BEST_ONNX), providers=["CPUExecutionProvider"])
ort_input = ort_session.get_inputs()[0]
ort_output = ort_session.get_outputs()[0]
print(f"ONNX 변환 완료: {BEST_ONNX} ({onnx_size_mb:.2f} MB, opset={ONNX_OPSET})")
print(f"- 입력 텐서: name={ort_input.name}, shape={ort_input.shape}, type={ort_input.type}")
print(f"- 출력 텐서: name={ort_output.name}, shape={ort_output.shape}, type={ort_output.type}")


def letterbox_preprocess(img_bgr: np.ndarray, imgsz: int = IMGSZ) -> tuple[np.ndarray, float, int, int, tuple[int, int]]:
    """Ultralytics Letterbox 규격에 맞춰 BGR 이미지를 NCHW float32 [1, 3, imgsz, imgsz] RGB 텐서로 전처리한다."""
    orig_h, orig_w = img_bgr.shape[:2]
    scale = min(imgsz / orig_h, imgsz / orig_w)
    new_w, new_h = int(round(orig_w * scale)), int(round(orig_h * scale))
    resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    dw = (imgsz - new_w) / 2.0
    dh = (imgsz - new_h) / 2.0
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(114, 114, 114)
    )

    rgb = cv2.cvtColor(padded, cv2.COLOR_BGR2RGB)
    tensor = np.ascontiguousarray(rgb.transpose(2, 0, 1)[None], dtype=np.float32) / 255.0
    return tensor, scale, left, top, (orig_h, orig_w)


def nms_xyxy(boxes: np.ndarray, scores: np.ndarray, iou_thres: float = NMS_IOU) -> list[int]:
    """순수 NumPy 기반 단일 클래스 Greedy NMS (iou_thres=NMS_IOU=0.45)."""
    if len(boxes) == 0:
        return []
    order = np.argsort(-scores)
    keep = []
    while order.size > 0:
        i = int(order[0])
        keep.append(i)
        if order.size == 1:
            break
        rest = order[1:]
        ious = compute_iou_matrix(boxes[i : i + 1], boxes[rest])[0]
        order = rest[ious <= iou_thres]
    return keep


def run_pure_onnx_inference(
    session: ort.InferenceSession,
    img_path: Path,
    imgsz: int = IMGSZ,
    conf_thres: float = SELECTED_CONF,
    iou_thres: float = NMS_IOU,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """순수 onnxruntime으로 추론하고 conf 필터링 + NMS(iou_thres=0.45) + 원본 좌표 복원을 수행한다."""
    img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    inp_tensor, scale, pad_x, pad_y, (orig_h, orig_w) = letterbox_preprocess(img_bgr, imgsz=imgsz)
    inp_name = session.get_inputs()[0].name
    raw_out = session.run(None, {inp_name: inp_tensor})[0]  # [1, 5, 8400]

    preds = raw_out[0].T  # [8400, 5]: cx, cy, w, h, conf
    scores = preds[:, 4]
    mask = scores >= conf_thres
    if not np.any(mask):
        return raw_out, np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32), inp_tensor

    sel = preds[mask]
    sel_scores = scores[mask]
    boxes_xyxy = np.empty((len(sel), 4), dtype=np.float32)
    boxes_xyxy[:, 0] = sel[:, 0] - sel[:, 2] / 2.0
    boxes_xyxy[:, 1] = sel[:, 1] - sel[:, 3] / 2.0
    boxes_xyxy[:, 2] = sel[:, 0] + sel[:, 2] / 2.0
    boxes_xyxy[:, 3] = sel[:, 1] + sel[:, 3] / 2.0

    keep_idx = nms_xyxy(boxes_xyxy, sel_scores, iou_thres=iou_thres)
    final_boxes = boxes_xyxy[keep_idx].copy()
    final_scores = sel_scores[keep_idx].copy()

    # Letterbox 역변환 및 원본 해상도 경계 클리핑
    final_boxes[:, [0, 2]] = np.clip((final_boxes[:, [0, 2]] - pad_x) / scale, 0.0, float(orig_w))
    final_boxes[:, [1, 3]] = np.clip((final_boxes[:, [1, 3]] - pad_y) / scale, 0.0, float(orig_h))
    return raw_out, final_boxes, final_scores, inp_tensor


# 2) 동일 입력 검증 영상 선택 (문서 검증 영상 우선, 없으면 양성 샘플 1장)
preferred_sample = LOCAL_DATA_DIR / "images" / "test" / "01e8ea3c498eb96bf39a1f019680aa7f.png"
if preferred_sample.exists():
    verify_img_path = preferred_sample
else:
    pos_candidates = [r["image_path"] for r in test_records if len(r["gt_boxes"]) > 0]
    verify_img_path = pos_candidates[0] if pos_candidates else test_records[0]["image_path"]

# 순수 ONNX Runtime 추론 (conf=SELECTED_CONF, iou_thres=NMS_IOU=0.45)
onnx_raw, onnx_boxes, onnx_scores, shared_inp = run_pure_onnx_inference(
    ort_session, verify_img_path, imgsz=IMGSZ, conf_thres=SELECTED_CONF, iou_thres=NMS_IOU
)

# PyTorch 동일 입력 원시 출력 텐서 비교
pt_net = best_model.model.to("cpu").eval()
with torch.no_grad():
    pt_raw_out = pt_net(torch.from_numpy(shared_inp))
pt_raw = (pt_raw_out[0] if isinstance(pt_raw_out, (tuple, list)) else pt_raw_out).cpu().numpy()
raw_max_abs_diff = float(np.max(np.abs(pt_raw - onnx_raw)))

# PyTorch 최종 검출 결과 비교 (conf=SELECTED_CONF, iou=NMS_IOU=0.45)
pt_res = best_model.predict(
    source=str(verify_img_path),
    imgsz=IMGSZ,
    conf=SELECTED_CONF,
    iou=NMS_IOU,
    device="cpu",
    verbose=False,
)[0]
pt_boxes = pt_res.boxes.xyxy.cpu().numpy().astype(np.float32) if pt_res.boxes is not None and len(pt_res.boxes) else np.zeros((0, 4), dtype=np.float32)
pt_scores = pt_res.boxes.conf.cpu().numpy().astype(np.float32) if pt_res.boxes is not None and len(pt_res.boxes) else np.zeros((0,), dtype=np.float32)

count_match = len(pt_boxes) == len(onnx_boxes)
onnx_verification = {
    "onnx_file": BEST_ONNX.name,
    "onnx_size_mb": round(onnx_size_mb, 2),
    "opset": ONNX_OPSET,
    "imgsz": IMGSZ,
    "input_name": ort_input.name,
    "input_shape": list(ort_input.shape),
    "output_name": ort_output.name,
    "output_shape": list(ort_output.shape),
    "verify_image": verify_img_path.name,
    "conf_thres": SELECTED_CONF,
    "nms_iou": NMS_IOU,
    "raw_output_max_abs_diff": round(raw_max_abs_diff, 6),
    "pytorch_detections": int(len(pt_boxes)),
    "onnxruntime_detections": int(len(onnx_boxes)),
    "detection_count_match": bool(count_match),
}

print(f"\n=== PyTorch vs 순수 ONNX Runtime 일치 검증 ({verify_img_path.name}) ===")
print(f"- 적용 기준: conf={SELECTED_CONF:.2f}, NMS iou_thres={NMS_IOU:.2f}")
print(f"- 원시 출력 텐서 최대 절대 오차 (max abs diff): {raw_max_abs_diff:.6f}")
print(f"- 최종 검출 건수: PyTorch {len(pt_boxes)}건 vs ONNX Runtime {len(onnx_boxes)}건 (일치={count_match})")
if len(pt_boxes) and len(onnx_boxes) and count_match:
    print(f"- PyTorch scores: {np.round(pt_scores, 4).tolist()}")
    print(f"- ONNX    scores: {np.round(onnx_scores, 4).tolist()}")

## 9. 가중치(`best.pt`, `best.onnx`) 및 평가 지표 JSON/CSV 저장 (Google Drive 백업 + 로컬 다운로드)

In [ ]:
from datetime import datetime
import json

summary_payload = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "run_name": RUN_NAME,
    "model": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "nms_iou": NMS_IOU,
    "match_iou": MATCH_IOU,
    "default_conf": DEFAULT_CONF,
    "selected_conf_from_val": SELECTED_CONF,
    "dataset_summary": summary_rows,
    "official_ultralytics_metrics": official_metrics,
    "eval_split": EVAL_SPLIT,
    "val_selected_conf_metrics": val_best_row,
    "test_official_fixed_conf_metrics": test_official_row,
    "test_posthoc_exploratory_best_f1_metrics": test_posthoc_best_row,
    "onnx_verification": onnx_verification,
}

metrics_json_path = RUN_DIR / "metrics_summary.json"
val_sweep_csv_path = RUN_DIR / "val_threshold_sweep.csv"
test_sweep_csv_path = RUN_DIR / "test_threshold_sweep_exploratory.csv"

metrics_json_path.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")
if not df_val_sweep.empty:
    df_val_sweep.to_csv(val_sweep_csv_path, index=False, encoding="utf-8-sig")
df_test_sweep.to_csv(test_sweep_csv_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {metrics_json_path}")
if val_sweep_csv_path.exists():
    print(f"저장 완료: {val_sweep_csv_path}")
print(f"저장 완료: {test_sweep_csv_path}")

# Google Drive에 주요 결과물 복사
if IN_COLAB:
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    files_to_backup = [
        BEST_WEIGHTS,
        BEST_ONNX,
        RUN_DIR / "weights" / "last.pt",
        RUN_DIR / "results.csv",
        RUN_DIR / "results.png",
        metrics_json_path,
        val_sweep_csv_path,
        test_sweep_csv_path,
        RUN_DIR / "viz_tp_success.png",
        RUN_DIR / "viz_fn_missed.png",
        RUN_DIR / "viz_fp_normal.png",
    ]
    for src_f in files_to_backup:
        if src_f.exists():
            shutil.copy2(src_f, DRIVE_SAVE_DIR / src_f.name)
    print(f"\n✅ Google Drive 백업 완료: {DRIVE_SAVE_DIR}")
    print("백업된 파일 목록:", [p.name for p in sorted(DRIVE_SAVE_DIR.iterdir())])

print("\n=== 최종 JSON 요약 ===")
print(metrics_json_path.read_text(encoding="utf-8"))

In [ ]:
# (선택) 브라우저에서 바로 best.pt, best.onnx와 평가 지표 파일 다운로드
if IN_COLAB:
    from google.colab import files
    for p in (BEST_WEIGHTS, BEST_ONNX, metrics_json_path, val_sweep_csv_path, test_sweep_csv_path):
        if p.exists():
            files.download(str(p))